<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%9E%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'



Mounted at /content/drive


In [2]:
# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    return df

orders_df = load_orders()


In [3]:
# Шаг 3. Загрузка тестовых пользователей
import glob

def load_test_users():
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in test_files:
        users.update(pd.read_parquet(f)['user_id'].unique())
    return list(users)

test_users = load_test_users()


In [4]:
# Шаг 4. Аналитическая "модель популярности" по всем покупкам за последние 2 недели

def analytic_popularity_order(orders_df, top_k=100, start_date='2025-07-02', end_date='2025-07-15'):
    # Только доставленные заказы за финальный период
    orders = orders_df[
        (orders_df['last_status']=='delivered_orders') &
        (orders_df['created_date'] >= pd.to_datetime(start_date)) &
        (orders_df['created_date'] <= pd.to_datetime(end_date))
    ]
    # Формируем user_preferences: порядок покупок
    orders = orders.sort_values(['user_id', 'created_timestamp'])
    user_prefs = {}
    for uid, group in orders.groupby('user_id'):
        items = group['item_id'].tolist()
        user_prefs[str(uid)] = items
    # Выбираем ТОП-K товаров по частоте (можно без сортировки по позициям)
    top_items = orders['item_id'].value_counts().head(top_k).index.tolist()
    return top_items

popular_items = analytic_popularity_order(orders_df, top_k=100)


In [5]:
# Шаг 5. Вычисление истории покупок для тестовых пользователей (только delivered)
def build_user_prefs(orders_df, test_users):
    delivered = orders_df[orders_df['last_status']=='delivered_orders']
    user_items = delivered.groupby('user_id')['item_id'].apply(set).to_dict()
    user_prefs = {}
    for uid in test_users:
        user_prefs[uid] = list(user_items.get(uid, set()))
    return user_prefs

user_preferences = build_user_prefs(orders_df, test_users)


In [6]:
# Шаг 6. Генерация рекомендаций (ТОП-100, без купленных товаров)
def generate_recommendations(test_users, popular_items, user_preferences):
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:100]
    return recs

recommendations = generate_recommendations(test_users, popular_items, user_preferences)


Генерация рекомендаций: 100%|██████████| 470347/470347 [00:05<00:00, 81155.83it/s] 


In [7]:
# Шаг 7. Формирование submission-файла (готово к сабмиту)
def save_submission(recommendations, filename='submission.csv'):
    rows = []
    for uid, items in tqdm(recommendations.items(), desc='Формирование submission'):
        rows.append({
            'user_id': uid,
            'item_id_1 item_id_2 ... item_id_100': ' '.join(map(str, items))
        })
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"Готово: {filename}")

save_submission(recommendations, filename='ozon_baseline_analytic_submission.csv')


Формирование submission: 100%|██████████| 470347/470347 [00:10<00:00, 46230.45it/s]


Готово: ozon_baseline_analytic_submission.csv
